# 01 Data Sync and Check

**Project**: AIGC Research Comprehension System  
**Purpose**: Prepare remote Google Drive storage, acquire the 100-paper corpus, and parse PDFs into evidence sections.

**Operational Strategy**:
- **Google Drive** is the ONLY persistent data store.
- **Colab** is the temporary compute runtime.
- **Safe to Rerun**: Notebook detects existing data and only performs missing work unless `RESET_DATA` is true.

In [ ]:
# @title Configuration Flags
DATA_PATH = "/content/drive/MyDrive/AIGC/Data_V2" # @param {type:"string"}
BRANCH = "day9-datav2-pipeline-fix" # @param {type:"string"}
RESET_DATA = False # @param {type:"boolean"}
RUN_DOWNLOAD = True # @param {type:"boolean"}
RUN_PARSE = True # @param {type:"boolean"}

MAX_PDF_TARGET = 100
MIN_PDF_REQUIRED_FOR_PARSE = 90
STOP_AFTER_PDF_COUNT = 100

# Optional: Set your email for Unpaywall API
UNPAYWALL_EMAIL = "" # @param {type:"string"}

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive", force_remount=True)
assert os.path.exists("/content/drive"), "Drive mount failed!"

In [ ]:
# === Global Project Configuration ===
PROJECT_NAME = "AIGC_Fake_Detection"
GITHUB_REPO = "https://github.com/IanJ332/AIGC_Fake_Detection.git"
BRANCH = "day9-datav2-pipeline-fix"

DRIVE_ROOT = "/content/drive/MyDrive/AIGC"
DATA_DIR = f"{DRIVE_ROOT}/Data_V2"
NOTEBOOK_DIR = f"{DRIVE_ROOT}/Notebook"

MANIFEST_PATH = "corpus/manifest.csv"

RUN_DOWNLOAD = True
RUN_PARSE = True
FORCE_REBUILD = False


## 2. Define Paths

In [ ]:
BASE = Path("/content/drive/MyDrive/AIGC")
DATA_DIR = Path(DATA_PATH)

PDF_DIR = DATA_DIR / "pdfs"
TEI_DIR = DATA_DIR / "tei_xml"
DOWNLOAD_LOG_DIR = DATA_DIR / "download_logs"
PARSED_DIR = DATA_DIR / "parsed"
SECTIONS_DIR = DATA_DIR / "sections"
TABLES_DIR = DATA_DIR / "tables"
PARSE_LOG_DIR = DATA_DIR / "parse_logs"
PROBES_DIR = DATA_DIR / "probes"
REGISTRY_DIR = DATA_DIR / "registry"
REPORTS_DIR = DATA_DIR / "reports"
CHECKPOINT_DIR = DATA_DIR / "checkpoints"

REPO_URL = "https://github.com/IanJ332/AIGC_Fake_Detection"
REPO_DIR = Path("/content/AIGC_Fake_Detection")

## 3. Folder Management

In [ ]:
import shutil

sub_dirs = [
    PDF_DIR, TEI_DIR, DOWNLOAD_LOG_DIR, PARSED_DIR, SECTIONS_DIR,
    TABLES_DIR, PARSE_LOG_DIR, PROBES_DIR, REGISTRY_DIR, REPORTS_DIR,
    CHECKPOINT_DIR
]

if RESET_DATA:
    print(f"RESET_DATA is True. Deleting and recreating {DATA_DIR}...")
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    for d in sub_dirs:
        d.mkdir(parents=True, exist_ok=True)
else:
    for d in sub_dirs:
        d.mkdir(parents=True, exist_ok=True)

print("Folders verified.")

## 4. Resource Report

In [ ]:
import sys
import psutil
import subprocess

print(f"Python: {sys.version}")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
    print(f"GPU: {gpu}")
except:
    print("GPU: None")

## 5. Clone Repository

In [ ]:
!rm -rf /content/AIGC_Fake_Detection
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
!pip install -q pandas tqdm pyyaml pymupdf requests

## 6. Sync Metadata & Setup Symlinks

In [ ]:
def sync_to_drive():
    # Copy lightweight files from repo to Drive
    mapping = {
        REPO_DIR / "corpus/manifest.csv": REGISTRY_DIR / "manifest_100.csv",
        REPO_DIR / "corpus/manifest.csv": REGISTRY_DIR / "manifest.csv"
    }
    for src, dst in mapping.items():
        if src.exists():
            shutil.copy(src, dst)

def setup_symlinks():
    data_subsets = [
        ("pdfs", PDF_DIR), ("tei_xml", TEI_DIR), ("download_logs", DOWNLOAD_LOG_DIR),
        ("parsed", PARSED_DIR), ("sections", SECTIONS_DIR), ("tables", TABLES_DIR),
        ("parse_logs", PARSE_LOG_DIR)
    ]
    for name, drive_path in data_subsets:
        local_path = REPO_DIR / "corpus" / name
        if local_path.exists() and not local_path.is_symlink():
            shutil.rmtree(local_path)
        if not local_path.exists():
            os.symlink(drive_path, local_path)

sync_to_drive()
setup_symlinks()
print("Metadata synced and symlinks established.")

## 7. Primary Acquisition

In [ ]:
if RUN_DOWNLOAD:
    %cd {REPO_DIR}
    # Use the new executable manifest
    !python -m src.acquire.download_documents --manifest corpus/manifest.csv
    !python -m src.acquire.validate_documents --registry corpus/document_registry.csv

    # Sync results back to Drive
    shutil.copy(REPO_DIR / "corpus/document_registry.csv", REGISTRY_DIR / "document_registry.csv")
    if (REPO_DIR / "docs/day2_document_acquisition_report.md").exists():
        shutil.copy(REPO_DIR / "docs/day2_document_acquisition_report.md", REPORTS_DIR / "day2_document_acquisition_report.md")
    print("Acquisition complete.")

## 8. Full PDF Parse

In [ ]:
import pandas as pd
import time
import json

reg_path = REGISTRY_DIR / "document_registry.csv"
pdf_count = 0
if reg_path.exists():
    df = pd.read_csv(reg_path)
    pdf_count = int(df["pdf_downloaded"].fillna(False).sum())

if RUN_PARSE and pdf_count >= MIN_PDF_REQUIRED_FOR_PARSE:
    %cd {REPO_DIR}
    !python -m src.parse.parse_pdfs --registry corpus/document_registry.csv
    !python -m src.parse.segment_sections --parsed-dir corpus/parsed
    !python -m src.parse.extract_table_candidates --sections corpus/sections/sections.jsonl
    !python -m src.parse.validate_parse

    # Sync results back to Drive
    if (REPO_DIR / "corpus/parse_registry.csv").exists():
        shutil.copy(REPO_DIR / "corpus/parse_registry.csv", REGISTRY_DIR / "parse_registry.csv")
    if (REPO_DIR / "docs/day3_parse_report.md").exists():
        shutil.copy(REPO_DIR / "docs/day3_parse_report.md", REPORTS_DIR / "day3_parse_report.md")
    print("Parsing complete.")
elif RUN_PARSE:
    print(f"RUN_PARSE is True but PDF count ({pdf_count}) is below threshold ({MIN_PDF_REQUIRED_FOR_PARSE}).")

## 9. Final Readiness Check

In [ ]:
import json
import glob

pdf_count = len(glob.glob(f"{DATA_DIR}/pdfs/*.pdf"))
parsed_json_count = len(glob.glob(f"{DATA_DIR}/parsed/*.json"))
sections_exists = os.path.exists(f"{DATA_DIR}/sections/sections.jsonl")
tables_exists = os.path.exists(f"{DATA_DIR}/tables/table_candidates.jsonl")

if parsed_json_count >= 95:
    status = "READY_FOR_NOTEBOOK_02"
elif 70 <= parsed_json_count < 95:
    status = "CAUTION_READY_FOR_NOTEBOOK_02"
else:
    status = "BLOCKED"

manifest_rows = len(open(MANIFEST_PATH).readlines()) - 1

print(f"Branch: {BRANCH}")
print(f"Data Dir: {DATA_DIR}")
print(f"Manifest Path: {MANIFEST_PATH}")
print(f"Manifest Rows: {manifest_rows}")
print(f"PDF Count: {pdf_count}")
print(f"Parsed JSON Count: {parsed_json_count}")
print(f"Sections Exists: {sections_exists}")
print(f"Tables Exists: {tables_exists}")
print(f"Final Readiness Status: {status}")

probe_status = {
    'branch': BRANCH,
    'data_dir': DATA_DIR,
    'manifest_path': MANIFEST_PATH,
    'manifest_rows': manifest_rows,
    'pdf_count': pdf_count,
    'parsed_json_count': parsed_json_count,
    'sections_exists': sections_exists,
    'tables_exists': tables_exists,
    'status': status
}

with open(f"{DATA_DIR}/probes/day9_data_status.json", "w") as f:
    json.dump(probe_status, f, indent=2)

with open(f"{DATA_DIR}/reports/day9_data_sync_report.md", "w") as f:
    f.write(f"# Day 9 Data Sync Report\n\nStatus: **{status}**\n")
    f.write(f"- PDFs: {pdf_count}\n- Parsed: {parsed_json_count}\n")
